# NutriAI - Machine Learning Pipeline Showcase

This notebook showcases the training, evaluation, and inference of the three primary Machine Learning models powering the NutriAI platform:
1. **XGBoost Classifier** (Clinical Diet Recommendation)
2. **Random Forest Regressor** (Dynamic Macro Target Calculation)
3. **K-Nearest Neighbors (KNN)** (Content-Based Meal Recommender)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## 1. XGBoost Diet Classifier
Predicts the ideal clinical diet (Balanced, Low-Carb, Low-Sodium) based on 18 physiological biomarkers.

In [ ]:
# Load Data
df_diet = pd.read_csv('datasets/diet_recommendations_dataset.csv')
df_diet = df_diet.drop(columns=['Patient_ID'], errors='ignore')

X_diet = df_diet.drop(columns=['Diet_Recommendation'])
y_diet = df_diet['Diet_Recommendation']

# Encode Categorical
for col in X_diet.select_dtypes(include=['object']).columns:
    X_diet[col] = LabelEncoder().fit_transform(X_diet[col])
    
y_encoded = LabelEncoder().fit_transform(y_diet)

# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X_diet, y_encoded, test_size=0.2, random_state=42)

# Train Model
xgb = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)

# Evaluate
y_pred = xgb.predict(X_test)
print("--- XGBoost Diet Classifier Performance ---")
print(classification_report(y_test, y_pred, target_names=df_diet['Diet_Recommendation'].unique()))


## 2. Random Forest Macro Regressor
Dynamically calculates the patient's Basal Metabolic Rate (Calories) and Protein RDA (g/day) based on Age and Weight.

In [ ]:
# Load Data
df_macro = pd.read_csv('datasets/final_nutrition_dataset.csv')
X_macro = df_macro[['Age (years)', 'Weight_Energy']]
y_macro = df_macro[['Energy Requirement (kcal/day)', 'RDA (g/day)']]

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(X_macro, y_macro, test_size=0.2, random_state=42)

# Train Multi-Output Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_m_train, y_m_train)

# Evaluate
y_m_pred = rf.predict(X_m_test)
print("--- Random Forest Macro Regressor Performance ---")
print(f"MAE Calories: {mean_absolute_error(y_m_test.iloc[:, 0], y_m_pred[:, 0]):.2f} kcal")
print(f"MAE Protein:  {mean_absolute_error(y_m_test.iloc[:, 1], y_m_pred[:, 1]):.2f} g")
print(f"R-Squared:    {r2_score(y_m_test, y_m_pred):.4f}")


## 3. K-Nearest Neighbors Meal Recommender
Uses a vector space of nutritional macros to find exact matches for the predicted calories and protein.

In [ ]:
# Load Data
df_meal = pd.read_csv('datasets/Food_and_Nutrition__.csv')
num_features = ['Calories', 'Protein', 'Carbohydrates', 'Fat', 'Sodium']

X_meal = df_meal[num_features].fillna(0)

# Standardise Macros
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_meal)

# Train KNN Space
knn = NearestNeighbors(n_neighbors=1, metric='euclidean')
knn.fit(X_scaled)

# Mock Inference for a Patient needing 2100 kcal and 120g protein
patient_query = np.array([[2100, 120, 250, 70, 2300]])
query_scaled = scaler.transform(patient_query)

distances, indices = knn.kneighbors(query_scaled)
best_match = df_meal.iloc[indices[0][0]]

print("--- KNN Recommended Day Plan ---")
print(f"Target   -> Cal: 2100 | Pro: 120g")
print(f"Matched  -> Cal: {best_match['Calories']} | Pro: {best_match['Protein']}g")
print(f"Breakfast: {best_match['Breakfast Suggestion']}")
print(f"Lunch:     {best_match['Lunch Suggestion']}")
print(f"Dinner:    {best_match['Dinner Suggestion']}")
